# 02.02 · Linear & logistic regression

**Chapter:** Machine Learning Fundamentals · **Milestone:** M1

## Goals
- [ ] Implement linear regression with gradient descent and compare with the closed form
- [ ] Implement logistic regression from scratch (sigmoid + cross-entropy)
- [ ] Reproduce both with scikit-learn

## Resources
- [Andrew Ng – ML Specialization, course 1](https://www.coursera.org/specializations/machine-learning-introduction) — free audit
- [Google MLCC – Linear & logistic regression](https://developers.google.com/machine-learning/crash-course) — free

These two models are the "hello world" of ML **and** the last layer of every neural net:
an LLM's final layer is a linear map + softmax, i.e. multinomial logistic regression over the vocabulary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Linear regression
Model: $\hat y = Xw + b$. Loss: mean squared error $L = \frac1n\sum(\hat y - y)^2$.
Gradients: $\partial L/\partial w = \frac2n X^T(\hat y - y)$, $\partial L/\partial b = \frac2n\sum(\hat y - y)$.

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)  # fit on train only
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)


def fit_linear_gd(X, y, lr=0.1, steps=20_000):
    w, b = np.zeros(X.shape[1]), 0.0
    losses = []
    for _ in range(steps):
        err = X @ w + b - y
        losses.append((err**2).mean())
        w -= lr * 2 * X.T @ err / len(y)
        b -= lr * 2 * err.mean()
    return w, b, losses


w, b, losses = fit_linear_gd(X_train, y_train)
sk = LinearRegression().fit(X_train, y_train)

def r2(y_true, y_pred):
    return 1 - ((y_true - y_pred) ** 2).sum() / ((y_true - y_true.mean()) ** 2).sum()

print("max |w_gd - w_sklearn|:", np.abs(w - sk.coef_).max().round(3))
print(f"test R²  scratch: {r2(y_test, X_test @ w + b):.3f} | sklearn: {sk.score(X_test, y_test):.3f}")
plt.plot(losses); plt.xscale("log"); plt.title("training MSE"); plt.show()

Why 20,000 steps? Some diabetes features are strongly correlated, so the loss surface is an elongated bowl (01.05).
With 2,000 steps the **loss** is already near-optimal but the **weights** are still far from sklearn's. Try it.
Closed-form solvers (sklearn uses least squares) don't have this problem; for neural nets there is no closed form,
hence better optimisers.

## 2. Logistic regression
For binary labels: $p = \sigma(Xw + b)$ with $\sigma(z) = 1/(1+e^{-z})$.
Loss: binary cross-entropy $-\frac1n\sum[y\log p + (1-y)\log(1-p)]$ (same idea as the LLM loss in 01.04).
Beautiful fact: the gradient has the same form as linear regression: $\partial L/\partial w = \frac1n X^T(p - y)$.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def fit_logistic(X, y, lr=0.1, steps=3000, l2=0.0):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * (p - y).mean()
    return w, b


def bce(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()


w, b = fit_logistic(X_train, y_train)
p_test = sigmoid(X_test @ w + b)
sk = LogisticRegression(max_iter=5000).fit(X_train, y_train)
print(f"scratch: acc={((p_test > 0.5) == y_test).mean():.3f} loss={bce(y_test, p_test):.3f}")
print(f"sklearn: acc={sk.score(X_test, y_test):.3f} loss={bce(y_test, sk.predict_proba(X_test)[:, 1]):.3f}")

Small differences come from regularisation: scikit-learn's `LogisticRegression` applies L2 by default (`C=1.0`).

## 3. Probabilities, not just labels
Logistic regression outputs **calibrated-ish probabilities**. You choose the threshold depending on cost of errors (02.03).
Same with LLMs: asking for a probability in text is unreliable; using **token logprobs** (04.06) is better.

In [ ]:
plt.hist(p_test[y_test == 0], bins=20, alpha=0.6, label="malignant (0)")
plt.hist(p_test[y_test == 1], bins=20, alpha=0.6, label="benign (1)")
plt.axvline(0.5, color="k", ls="--"); plt.legend(); plt.title("predicted P(benign)"); plt.show()

## 4. Multiclass = softmax regression
With K classes: $K$ weight vectors, softmax over the $K$ scores, cross-entropy loss. Swap "classes" for "50k vocabulary tokens"
and this is an LLM's output head.

## Exercises
1. Use `fit_logistic(..., l2=λ)` for λ in [0, 0.001, 0.01, 0.1, 1]. Plot test accuracy and `‖w‖` vs λ.
2. Implement **softmax regression** from scratch on `sklearn.datasets.load_digits` (10 classes). Target: >95% test accuracy.
   Check gradients numerically on a tiny subset first.
3. Replace full-batch gradient descent with **mini-batch SGD** (batch size 32). Compare convergence speed.

In [ ]:
# your code here